# MLOps: Experiment Tracking

You have reached the grand finale of the Machine Learning Fundamentals curriculum! 

We know how to build models, tune them, mathematically audit them, and serialize them. But there is one final, catastrophic trap that catches almost every Junior Data Scientist. 

When you run a Grid Search, you might train 500 different models. Next week, you get a new dataset and train 500 more. A month later, your boss asks: *"Hey, remember that Random Forest that got 92% accuracy three weeks ago? What exact learning rate did you use? What exact dataset version did it see?"*

If your answer is *"I think I wrote it down on a sticky note,"* your model cannot be deployed. In the enterprise world, an algorithm is a liability unless it is **100% reproducible**. In this final lesson, we explore how MLOps engineers track, version, and statistically compare thousands of models using **Experiment Tracking**.

Experiment Tracking is the process of saving all the metadata, inputs, outputs, and files associated with a machine learning training run into a centralized database. The absolute industry standard for this is **MLflow** (created by Databricks) or **Weights & Biases (W&B)**.

Let's set up our Python environment to simulate building a professional ML tracking database.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, log_loss

# In a real environment, you would: import mlflow
# For this notebook, we will simulate its core logic.

# Set professional visualization styling
sns.set_theme(style="whitegrid")

print("✅ MLOps Experiment Tracking Environment Ready.")

✅ MLOps Experiment Tracking Environment Ready.


# 1. The Anatomy of a "Run"

In MLOps, every single time you hit "execute" on a training script, it is called a **Run**. To guarantee that a Run is reproducible, your tracking system must mathematically lock down three distinct pillars of information:

### Pillar 1: Parameters (The Inputs)
These are the Hyperparameters ($\alpha, K, \nu$) and configuration settings you chose *before* the model started training. If you do not track these, you can never reconstruct the model's architecture.

### Pillar 2: Metrics (The Outputs)
These are the mathematical evaluations calculated *after* the model finishes training. You must track multiple metrics (Accuracy, F1-Score, Brier Score, Training Time). If you only track one metric, you will fall victim to the traps we discussed in Lesson 15.

### Pillar 3: Artifacts (The Files)
These are the physical files generated during the Run. This includes the frozen `.joblib` model file (Lesson 21), but it also includes diagnostic images like the ROC Curve, the Confusion Matrix, and the SHAP Waterfall plots. 

# 2. Implementing MLflow Logic in Code

Let's simulate a hyperparameter search. Instead of letting `RandomizedSearchCV` run blindly and throw away the results of its failed attempts, we will wrap our training loop in an Experiment Tracker. We want a permanent record of every success and every failure.

In [2]:
# 1. Simulate a Corporate Dataset
X, y = make_classification(n_samples=2000, n_features=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Setup a simulated MLflow Database (A Pandas DataFrame)
experiment_db = []

# 3. Define our Hyperparameter Search Space
learning_rates = [0.01, 0.1, 0.5]
depths = [3, 5, 10]

print("🚀 Starting MLOps Experiment Tracking Loop...")

run_id = 1
for lr in learning_rates:
    for depth in depths:
        # --- MLFLOW START RUN ---
        # In reality, you call: with mlflow.start_run():
        
        # Train the model
        model = GradientBoostingClassifier(learning_rate=lr, max_depth=depth, n_estimators=50, random_state=42)
        model.fit(X_train, y_train)
        
        # Calculate Metrics
        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)
        acc = accuracy_score(y_test, preds)
        loss = log_loss(y_test, probs)
        
        # --- MLFLOW LOGGING ---
        # mlflow.log_param("learning_rate", lr)
        # mlflow.log_metric("accuracy", acc)
        # mlflow.sklearn.log_model(model, "model_artifact")
        
        # Simulate saving to the database
        experiment_db.append({
            "Run_ID": f"RUN_{run_id:03d}",
            "Param_LearningRate": lr,
            "Param_MaxDepth": depth,
            "Metric_Accuracy": acc,
            "Metric_LogLoss": loss
        })
        run_id += 1

print("✅ All runs logged successfully to the tracking server.")

# 4. Querying the MLOps Database
df_runs = pd.DataFrame(experiment_db)
display(df_runs.sort_values(by="Metric_LogLoss", ascending=True).head())

🚀 Starting MLOps Experiment Tracking Loop...
✅ All runs logged successfully to the tracking server.


,Run_ID,Param_LearningRate,Param_MaxDepth,Metric_Accuracy,Metric_LogLoss
4,RUN_005,0.1,5,0.9275,0.181442
3,RUN_004,0.1,3,0.9375,0.191232
6,RUN_007,0.5,3,0.9275,0.210465
5,RUN_006,0.1,10,0.9225,0.321191
7,RUN_008,0.5,5,0.9200,0.348045


*(Insight: Look at the resulting database table. We have a permanent, queryable record of every experiment. If a stakeholder asks why we didn't use a Learning Rate of 0.5, we can instantly query the database and prove that `RUN_009` violently overfitted and resulted in a terrible Log-Loss score!)*

# 3. Ablation Studies & Statistical Rigor

Why is tracking failures just as important as tracking successes? Because of **Ablation Studies**.

An Ablation Study is a scientific process where you systematically remove or change exactly *one* component of your machine learning pipeline while holding everything else perfectly constant. 

For example, suppose you add a highly complex NLP sentiment analyzer to your pipeline, and your accuracy jumps by $2\%$. Did the complex NLP cause the jump, or was it just a lucky random seed? 
If you use an Experiment Tracker, you can filter your runs to find the exact two models that are 100% identical except for the NLP feature. You can then run a rigorous statistical test (like a paired t-test) on their Cross-Validation scores to mathematically prove that the new feature is statistically significant.

# 4. The ML System Architecture (Putting it all together)

You now have all the pieces of a modern, enterprise-grade Machine Learning system. If you were to draw a blueprint of everything we have learned, it looks like this:

1. **Problem Framing (L1)**: Define $X$ and $y$. Establish a naive statistical baseline.
2. **Data Pipeline (L19)**: Use `ColumnTransformer` to safely Impute and Scale the data without leakage.
3. **Algorithm Selection (L3-L13)**: Choose an algorithm (Logistic, SVM, Random Forest, XGBoost) based on the geometry of the problem.
4. **Optimization (L2, L16)**: Use Gradient Descent to find parameters ($\theta$), and Bayesian Optimization to find hyperparameters ($\alpha$).
5. **Validation (L14-L15)**: Use Stratified K-Fold CV to grade the model using business-specific metrics (Precision/Recall).
6. **Interpretability (L20)**: Use SHAP to audit the model and ensure it isn't making decisions based on illegal or biased features.
7. **Calibration (L18)**: Apply Isotonic Regression to ensure the probabilities are mathematically trustworthy.
8. **Serialization & Tracking (L21-L22)**: Log the metrics to MLflow, freeze the model using ONNX, and deploy it to a server.

## Real-World Use Case or Analogy:
Think of Experiment Tracking like a **Chemist's Laboratory Notebook**:

* **The Amateur**: Mixes random chemicals in a beaker until something blows up, then tries to remember exactly how many drops of acid they used. They can never recreate the explosion.
* **The MLOps Engineer**: Before adding a single drop, they write the date, the exact temperature of the room, and the manufacturer of the acid in their notebook (**Logging Parameters**). After the reaction, they measure the exact volume of gas produced (**Logging Metrics**). They seal a sample of the resulting chemical in a labeled glass vial and store it in a fridge (**Logging Artifacts**). If someone asks them to recreate the reaction 10 years later, they can do it flawlessly.

---